## Audio RAG — Whisper (Full-Text) vs CLAP (Shared Semantic Space)

Experiments on audio input (MP3 from a YouTube video about Transformers / Attention).

### Pipeline comparison

| Aspect | Whisper pipeline | CLAP pipeline |
|---|---|---|
| Indexing | Audio → Whisper transcription → text embedding | Audio segments → CLAP audio encoder → audio embedding |
| Query encoding | Text embedding model | CLAP **text** encoder (same shared space) |
| Retrieval | Text ↔ Text similarity | Text ↔ Audio cross-modal similarity |
| Context for LLM | Transcribed text chunks | Whisper transcription of **retrieved** audio segments |
| BM25 | On transcription | Not applicable |
| Qdrant collection | `QDRANT_WHISPER_COLLECTION` | `QDRANT_CLAP_COLLECTION` |

> **Key insight**: in both pipelines the LLM receives text. What differs is *how* the relevant
> segments are found: keyword/semantic similarity on transcription (Whisper) vs
> cross-modal similarity in the CLIP-style audio-text space (CLAP).


### Evaluation
The final section compares Whisper and CLAP RAG with BERTScore, answer precision/recall, context recall, timestamp coverage, and must/should claim recall. Evaluation generation is cached separately from the interactive filter table, and each approach exposes only retrieval modes that make sense for that pipeline.


### 1. Configuration


In [1]:
import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv("setup.env", override=True)

# ── Audio input ───────────────────────────────────────────────────────────────
# Path to the MP3 file. Accepted formats: mp3, wav, m4a, flac.
# Tip: download from YouTube with:
#   yt-dlp -x --audio-format mp3 -o "content/audio.mp3" <URL>
AUDIO_PATH = os.getenv("AUDIO_PATH", "./content/audio.mp3")

# ── Whisper (Speech-to-Text) ──────────────────────────────────────────────────
#
# Backends:
#   "faster-whisper"  — recommended, fast GPU inference via CTranslate2
#   "hf"              — fallback using transformers.pipeline
#
# Good whisper model choices:
#   "deepdml/faster-whisper-large-v3-turbo-ct2" — fast + high quality
#   "Systran/faster-whisper-large-v3"           — better quality, slower
#   "Systran/faster-whisper-medium"             — smaller/faster fallback
WHISPER_BACKEND = os.getenv("WHISPER_BACKEND", "faster-whisper")
WHISPER_MODEL = os.getenv("WHISPER_MODEL", "deepdml/faster-whisper-large-v3-turbo-ct2")
WHISPER_LANGUAGE = os.getenv("WHISPER_LANGUAGE", "en")   # set to empty/None for auto-detect
WHISPER_BATCH_SIZE = int(os.getenv("WHISPER_BATCH_SIZE", "16"))  # RTX 4070: 16 is a good start
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "cuda")         # "cuda" | "cpu" | "auto"
WHISPER_COMPUTE_TYPE = os.getenv("WHISPER_COMPUTE_TYPE", "float16") # use "int8_float16" if low VRAM
WHISPER_BEAM_SIZE = int(os.getenv("WHISPER_BEAM_SIZE", "1"))     # 1 = fastest; 5 = potentially better quality
WHISPER_VAD_FILTER = os.getenv("WHISPER_VAD_FILTER", "true").lower() == "true"

# ── Audio chunking ────────────────────────────────────────────────────────────
# Whisper returns segment-level timestamps. We merge consecutive segments
# into chunks of at most AUDIO_CHUNK_SECS seconds for indexing.
AUDIO_CHUNK_SECS  = int(os.getenv("AUDIO_CHUNK_SECS",   "45"))   # max seconds per text chunk
AUDIO_CHUNK_OVERLAP_SECS = int(os.getenv("AUDIO_CHUNK_OVERLAP_SECS", "5"))  # overlap between chunks

# CLAP splits the raw audio into fixed-length segments for embedding.
CLAP_SEGMENT_SECS = int(os.getenv("CLAP_SEGMENT_SECS",    "10"))  # seconds per CLAP segment
CLAP_SEGMENT_OVERLAP = int(os.getenv("CLAP_SEGMENT_OVERLAP",  "2"))  # overlap in seconds

# ── CLAP model ────────────────────────────────────────────────────────────────
# "laion/larger_clap_general"   — best general-purpose (music + speech + AudioSet)
# "laion/larger_clap_music"     — music-specialised
# "laion/clap-htsat-unfused"    — lighter baseline
CLAP_MODEL = os.getenv("CLAP_MODEL", "laion/larger_clap_general")
# CLAP_MODEL      = os.getenv("CLAP_MODEL", "laion/clap-htsat-unfused")

# ── Text embedding (Whisper pipeline) ─────────────────────────────────────────
# Same model as rag_pipeline_local.ipynb for fair comparison.
# "BAAI/bge-m3"                              — recommended (1024-dim)
# "sentence-transformers/all-MiniLM-L6-v2"  — lightweight (384-dim)
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVER_K = int(os.getenv("RETRIEVER_K",    "8"))
RERANKER_TOP_N = int(os.getenv("RERANKER_TOP_N", "4"))
RERANKER_MODEL = os.getenv("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
ENABLE_BM25 = os.getenv("ENABLE_BM25", "true").lower() == "true"
ENABLE_RERANKING  = os.getenv("ENABLE_RERANKING", "true").lower() == "true"

# ── Qdrant (local Docker) ─────────────────────────────────────────────────────
# docker run -d --name qdrant -p 6333:6333 -p 6334:6334 \
#   -v $(pwd)/qdrant_storage:/qdrant/storage qdrant/qdrant
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
QDRANT_WHISPER_COLLECTION = os.getenv("QDRANT_WHISPER_COLLECTION", "audio_whisper_v1")
QDRANT_CLAP_COLLECTION = os.getenv("QDRANT_CLAP_COLLECTION", "audio_clap_v1")
RESET_WHISPER_COLLECTION = os.getenv("RESET_WHISPER_COLLECTION", "false").lower() == "true"
RESET_CLAP_COLLECTION = os.getenv("RESET_CLAP_COLLECTION", "false").lower() == "true"

# ── Generation model ──────────────────────────────────────────────────────────
# Text-only generation (audio pipelines never pass raw audio to the LLM).
# "Qwen/Qwen2.5-3B-Instruct"          — local, CPU/GPU
# "Qwen/Qwen2.5-VL-3B-Instruct"       — also works (text-only prompt)
# Or use Ollama: set GENERATION_BACKEND=ollama and GENERATION_MODEL=mistral-nemo
GENERATION_MODEL = os.getenv("GENERATION_MODEL",   "Qwen/Qwen2.5-3B-Instruct")
GENERATION_BACKEND  = os.getenv("GENERATION_BACKEND", "hf")  # "hf" | "ollama"
GENERATION_MAX_NEW_TOKENS = int(os.getenv("GENERATION_MAX_NEW_TOKENS", "512"))
HF_DEVICE_MAP = os.getenv("HF_DEVICE_MAP",  "auto")
HF_TORCH_DTYPE = os.getenv("HF_TORCH_DTYPE", "auto")

# ── Persistence ───────────────────────────────────────────────────────────────
PERSIST_DIR = os.getenv("PERSIST_DIR", "./cache/audio/")
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs("./content/", exist_ok=True)

print("Configuration loaded.")
print(f"  Audio      : {AUDIO_PATH}")
print(f"  Whisper    : {WHISPER_MODEL} | backend={WHISPER_BACKEND} | lang={WHISPER_LANGUAGE or 'auto'}")
print(f"  Whisper HW : device={WHISPER_DEVICE}, compute={WHISPER_COMPUTE_TYPE}, batch={WHISPER_BATCH_SIZE}, beam={WHISPER_BEAM_SIZE}")
print(f"  Chunk      : {AUDIO_CHUNK_SECS}s (overlap={AUDIO_CHUNK_OVERLAP_SECS}s)")
print(f"  CLAP       : {CLAP_MODEL} | segment={CLAP_SEGMENT_SECS}s (overlap={CLAP_SEGMENT_OVERLAP}s)")
print(f"  Embeddings : {EMBEDDING_MODEL}")
print(f"  Retrieval  : k={RETRIEVER_K}, reranker_top_n={RERANKER_TOP_N}")
print(f"  Generator  : {GENERATION_MODEL} ({GENERATION_BACKEND})")
print(f"  Qdrant     : {QDRANT_URL}")
print(f"    Whisper collection : {QDRANT_WHISPER_COLLECTION}")
print(f"    CLAP collection    : {QDRANT_CLAP_COLLECTION}")


Configuration loaded.
  Audio      : ./content/audio.mp3
  Whisper    : deepdml/faster-whisper-large-v3-turbo-ct2 | backend=faster-whisper | lang=en
  Whisper HW : device=cuda, compute=float16, batch=16, beam=1
  Chunk      : 45s (overlap=5s)
  CLAP       : laion/larger_clap_general | segment=10s (overlap=2s)
  Embeddings : BAAI/bge-m3
  Retrieval  : k=8, reranker_top_n=4
  Generator  : Qwen/Qwen2.5-3B-Instruct (hf)
  Qdrant     : http://localhost:6333
    Whisper collection : audio_whisper_v1
    CLAP collection    : audio_clap_v1


### 2. Audio Loading

Loads the MP3 and converts it to a 48 kHz mono waveform for CLAP audio embeddings.

Whisper transcription with the recommended `faster-whisper` backend reads `AUDIO_PATH`
directly, so it does not depend on this 48 kHz waveform.

> If the audio file is not yet available, you can download it with:
> ```bash
> pip install yt-dlp
> yt-dlp -x --audio-format mp3 -o "content/audio.mp3" <YOUTUBE_URL>
> ```


In [2]:
import numpy as np
import librosa
from pathlib import Path

SAMPLE_RATE = 48000  # Hz — required by both Whisper and CLAP

def load_audio(path: str, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Load an audio file and return a normalised float32 mono waveform at `sr` Hz.
    Supports mp3, wav, flac, m4a via librosa/ffmpeg.
    """
    audio_path = Path(path)
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {path}\n"
            "Download with: yt-dlp -x --audio-format mp3 -o content/audio.mp3 <URL>"
        )
    waveform, _ = librosa.load(path, sr=sr, mono=True)
    print(f"Loaded: {audio_path.name}")
    print(f"  Duration : {len(waveform)/sr:.1f}s  ({len(waveform)/sr/60:.1f} min)")
    print(f"  Samples  : {len(waveform):,}  @ {sr} Hz")
    return waveform

waveform = load_audio(AUDIO_PATH)
AUDIO_DURATION_SECS = len(waveform) / SAMPLE_RATE


Loaded: audio.mp3
  Duration : 629.3s  (10.5 min)
  Samples  : 30,207,744  @ 48000 Hz


---
## 3. Whisper Pipeline — Full-Text Conversion

```
MP3 → Whisper → word-level segments + timestamps
    → merge into text chunks (≤ AUDIO_CHUNK_SECS)
    → text embedding (EMBEDDING_MODEL)
    → Qdrant QDRANT_WHISPER_COLLECTION
    → BM25 on transcription
    → hybrid retrieval → LLM (text only)
```


### 3.1 Transcription with Whisper

```bash
pip install faster-whisper
```

The notebook still supports the Hugging Face backend by setting:

```env
WHISPER_BACKEND=hf
WHISPER_MODEL=openai/whisper-large-v3
```


In [3]:
import torch

WHISPER_AVAILABLE = False
whisper_pipe = None
whisper_fw_model = None
whisper_fw_batched_model = None

def _resolve_whisper_device() -> str:
    """Resolve requested Whisper device, with a safe CPU fallback."""
    requested = (WHISPER_DEVICE or "auto").lower()
    if requested == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        print("  [warn] WHISPER_DEVICE=cuda requested, but CUDA is not available. Falling back to CPU.")
        return "cpu"
    return requested

whisper_device = _resolve_whisper_device()
print(f"Loading Whisper model: {WHISPER_MODEL} ({WHISPER_BACKEND}) ...")

if WHISPER_BACKEND.lower() in {"faster-whisper", "faster_whisper", "ct2"}:
    try:
        from faster_whisper import WhisperModel, BatchedInferencePipeline

        whisper_fw_model = WhisperModel(
            WHISPER_MODEL,
            device=whisper_device,
            compute_type=WHISPER_COMPUTE_TYPE,
        )
        whisper_fw_batched_model = BatchedInferencePipeline(model=whisper_fw_model)
        WHISPER_AVAILABLE = True
        print(
            f"  ✓ faster-whisper ready on {whisper_device} "
            f"(compute={WHISPER_COMPUTE_TYPE}, batch={WHISPER_BATCH_SIZE})"
        )
    except Exception as e:
        print(f"  ✗ Failed to load faster-whisper: {e}")
        print("    Install with: pip install faster-whisper")
        WHISPER_AVAILABLE = False

elif WHISPER_BACKEND.lower() in {"hf", "transformers", "huggingface"}:
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

    torch_dt = torch.float16 if whisper_device == "cuda" else torch.float32
    try:
        whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
        whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
            WHISPER_MODEL,
            torch_dtype=torch_dt,
            low_cpu_mem_usage=True,
        ).to(whisper_device)

        whisper_pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=whisper_model,
            tokenizer=whisper_processor.tokenizer,
            feature_extractor=whisper_processor.feature_extractor,
            torch_dtype=torch_dt,
            device=0 if whisper_device == "cuda" else -1,
            return_timestamps=True,
            chunk_length_s=30,
            batch_size=WHISPER_BATCH_SIZE,
            generate_kwargs={"language": WHISPER_LANGUAGE} if WHISPER_LANGUAGE else {},
        )
        WHISPER_AVAILABLE = True
        print(f"  ✓ Hugging Face Whisper ready on {whisper_device}")
    except Exception as e:
        print(f"  ✗ Failed to load Hugging Face Whisper: {e}")
        WHISPER_AVAILABLE = False

else:
    raise ValueError(
        "Unsupported WHISPER_BACKEND. Use 'faster-whisper' or 'hf'. "
        f"Got: {WHISPER_BACKEND!r}"
    )


Loading Whisper model: deepdml/faster-whisper-large-v3-turbo-ct2 (faster-whisper) ...
  ✓ faster-whisper ready on cuda (compute=float16, batch=16)


In [4]:
import json, hashlib, re
from pathlib import Path

# ── Transcription cache ───────────────────────────────────────────────────────
# Whisper on a long audio file is slow — cache to disk to avoid re-running.
def _audio_hash(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()[:12]

def _safe_cache_part(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(value)).strip("-")

_cache_name = "_".join([
    "transcript",
    _audio_hash(AUDIO_PATH),
    _safe_cache_part(WHISPER_BACKEND),
    _safe_cache_part(WHISPER_MODEL.split("/")[-1]),
    _safe_cache_part(WHISPER_COMPUTE_TYPE),
]) + ".json"
TRANSCRIPT_CACHE = Path(PERSIST_DIR) / _cache_name


def _transcribe_faster_whisper(audio_path: str) -> dict:
    """Transcribe AUDIO_PATH with faster-whisper and return HF-like {text, chunks}."""
    language = WHISPER_LANGUAGE or None
    segments_iter, info = whisper_fw_batched_model.transcribe(
        audio_path,
        batch_size=WHISPER_BATCH_SIZE,
        language=language,
        vad_filter=WHISPER_VAD_FILTER,
        beam_size=WHISPER_BEAM_SIZE,
        word_timestamps=False,
    )

    chunks = []
    texts = []
    for seg in segments_iter:
        text = seg.text.strip()
        if not text:
            continue
        chunks.append({"text": text, "timestamp": [float(seg.start), float(seg.end)]})
        texts.append(text)

    detected_language = getattr(info, "language", None)
    language_probability = getattr(info, "language_probability", None)
    if detected_language:
        print(f"  Detected language: {detected_language} ({language_probability:.2f})")

    return {
        "text": " ".join(texts).strip(),
        "chunks": chunks,
        "metadata": {
            "backend": "faster-whisper",
            "model": WHISPER_MODEL,
            "device": whisper_device,
            "compute_type": WHISPER_COMPUTE_TYPE,
            "batch_size": WHISPER_BATCH_SIZE,
            "beam_size": WHISPER_BEAM_SIZE,
            "vad_filter": WHISPER_VAD_FILTER,
            "detected_language": detected_language,
            "language_probability": language_probability,
        },
    }


def _transcribe_hf(waveform: np.ndarray) -> dict:
    """Transcribe waveform with the Hugging Face ASR pipeline."""
    result = whisper_pipe(waveform.copy(), return_timestamps=True)
    return {
        "text": result["text"],
        "chunks": [
            {"text": c["text"].strip(), "timestamp": list(c["timestamp"])}
            for c in result.get("chunks", [])
            if c.get("text", "").strip()
        ],
        "metadata": {
            "backend": "hf",
            "model": WHISPER_MODEL,
            "device": whisper_device,
        },
    }


def transcribe(waveform: np.ndarray, cache_path: Path, audio_path: str = AUDIO_PATH) -> dict:
    """Transcribe audio and return a common dict with 'text' and timestamped 'chunks'."""
    if cache_path.exists():
        print(f"Loading transcription from cache: {cache_path.name}")
        with open(cache_path, encoding="utf-8") as f:
            return json.load(f)

    if not WHISPER_AVAILABLE:
        raise RuntimeError("Whisper model not loaded.")

    print("Transcribing audio...")
    if WHISPER_BACKEND.lower() in {"faster-whisper", "faster_whisper", "ct2"}:
        serialisable = _transcribe_faster_whisper(audio_path)
    else:
        serialisable = _transcribe_hf(waveform)

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(serialisable, f, ensure_ascii=False, indent=2)
    print(f"  ✓ Transcription saved to cache: {cache_path.name}")
    return serialisable


transcription = transcribe(waveform, TRANSCRIPT_CACHE, AUDIO_PATH)
full_text = transcription["text"]
segments  = transcription["chunks"]   # list of {text, timestamp: [start, end]}

print(f"\nTranscription complete: {len(segments)} segments, {len(full_text)} chars")
print("\nFirst 500 chars:")
print(full_text[:500])


Transcribing audio...
  Detected language: en (1.00)
  ✓ Transcription saved to cache: transcript_2bad6f742512_faster-whisper_faster-whisper-large-v3-turbo-ct2_float16.json

Transcription complete: 23 segments, 9614 chars

First 500 chars:
BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models. BERT is designed to pre-trained deep bi-directional representations from unlabeled text by jointly conditioning on both left and right context in


### 3.2 Text Chunking with Timestamps


In [5]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class AudioChunk:
    """
    A chunk of transcribed audio with timestamp metadata.
    doc_type is always 'text' for both pipelines — the LLM only receives text.
    """
    content:    str            # transcribed text of this chunk
    doc_type:   str = "text"
    start_sec:  float = 0.0   # start time in the original audio
    end_sec:    float = 0.0   # end time in the original audio
    source_file: str = ""
    metadata:   dict = field(default_factory=dict)

    @property
    def timestamp_label(self) -> str:
        def fmt(s):
            m, sec = divmod(int(s), 60)
            return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"


def merge_segments_into_chunks(
    segments:     List[dict],
    max_secs:     int = AUDIO_CHUNK_SECS,
    overlap_secs: int = AUDIO_CHUNK_OVERLAP_SECS,
    source_file:  str = AUDIO_PATH,
) -> List[AudioChunk]:
    """
    Merge consecutive Whisper segments into chunks of at most `max_secs` seconds.
    Adds `overlap_secs` of context from the previous chunk to each new chunk.

    Each Whisper segment has: {"text": str, "timestamp": [start, end]}
    End may be None for the last segment — use audio duration as fallback.
    """
    if not segments:
        return []

    chunks:  List[AudioChunk] = []
    buf_text:  List[str]   = []
    buf_segs:  List[dict]  = []
    buf_start: float       = segments[0]["timestamp"][0] or 0.0

    def flush(buf_text, buf_segs, buf_start):
        if not buf_text:
            return
        text  = " ".join(buf_text).strip()
        end   = buf_segs[-1]["timestamp"][1] or AUDIO_DURATION_SECS
        chunks.append(AudioChunk(
            content    = text,
            start_sec  = buf_start,
            end_sec    = end,
            source_file= source_file,
        ))

    overlap_buf: List[dict] = []   # segments carried over for context

    for seg in segments:
        ts    = seg["timestamp"]
        start = ts[0] if ts[0] is not None else (buf_start if buf_segs else 0.0)
        end   = ts[1] if ts[1] is not None else AUDIO_DURATION_SECS

        # Start a new chunk if max duration is exceeded
        if buf_segs and (end - buf_start) > max_secs:
            flush(buf_text, buf_segs, buf_start)
            # Carry-over overlap segments
            overlap_buf = [s for s in buf_segs if (s["timestamp"][1] or AUDIO_DURATION_SECS) >= (buf_start + max_secs - overlap_secs)]
            buf_text  = [s["text"] for s in overlap_buf]
            buf_segs  = list(overlap_buf)
            buf_start = overlap_buf[0]["timestamp"][0] if overlap_buf else start

        buf_text.append(seg["text"])
        buf_segs.append(seg)

    flush(buf_text, buf_segs, buf_start)
    return chunks


whisper_chunks = merge_segments_into_chunks(segments)
print(f"Created {len(whisper_chunks)} text chunks from {len(segments)} Whisper segments.")
print(f"  avg duration : {sum(c.end_sec - c.start_sec for c in whisper_chunks)/len(whisper_chunks):.1f}s")
print(f"\nFirst chunk {whisper_chunks[0].timestamp_label}:")
print(whisper_chunks[0].content[:300])


Created 23 text chunks from 23 Whisper segments.
  avg duration : 27.4s

First chunk [00:00 – 00:25]:
BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Trans


### 3.3 Text Embedding + Qdrant


In [6]:
import uuid
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# ── Text embedding model ──────────────────────────────────────────────────────
print(f"Loading text embedding model: {EMBEDDING_MODEL} ...")
text_embedding = HuggingFaceEmbeddings(
    model_name    = EMBEDDING_MODEL,
    model_kwargs  = {"trust_remote_code": True},
    encode_kwargs = {"normalize_embeddings": True},
)
EMBEDDING_DIM = len(text_embedding.embed_query("dim probe"))
print(f"  ✓ Text embedding ready (dim={EMBEDDING_DIM})")

# ── Qdrant client (reusable across both pipelines) ────────────────────────────
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)

# ── Whisper collection ────────────────────────────────────────────────────────
if RESET_WHISPER_COLLECTION:
    client.delete_collection(QDRANT_WHISPER_COLLECTION)
    print(f"✓ Deleted '{QDRANT_WHISPER_COLLECTION}'")

if not client.collection_exists(QDRANT_WHISPER_COLLECTION):
    client.create_collection(
        collection_name = QDRANT_WHISPER_COLLECTION,
        vectors_config  = VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created Whisper collection '{QDRANT_WHISPER_COLLECTION}' (dim={EMBEDDING_DIM})")
else:
    print(f"✓ Using existing Whisper collection '{QDRANT_WHISPER_COLLECTION}'")

whisper_vector_store = QdrantVectorStore(
    client          = client,
    collection_name = QDRANT_WHISPER_COLLECTION,
    embedding       = text_embedding,
)

# ── Docstore: UUID → AudioChunk ──────────────────────────────────────────────
whisper_docstore: Dict[str, AudioChunk] = {}


Loading text embedding model: BAAI/bge-m3 ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  ✓ Text embedding ready (dim=1024)
✓ Created Whisper collection 'audio_whisper_v1' (dim=1024)


In [7]:
from langchain_core.documents import Document

def index_whisper_chunks(
    chunks:       List[AudioChunk],
    vector_store,
    docstore:     dict,
    batch_size:   int = 64,
) -> List[str]:
    """Index transcribed text chunks into Qdrant Whisper collection."""
    all_ids: List[str] = []
    lc_docs: List[Document] = []

    for chunk in chunks:
        if not chunk.content.strip():
            continue
        uid = str(uuid.uuid4())
        all_ids.append(uid)
        docstore[uid] = chunk
        lc_docs.append(Document(
            page_content = chunk.content,
            metadata     = {
                "doc_id":    uid,
                "doc_type":  "text",
                "start_sec": chunk.start_sec,
                "end_sec":   chunk.end_sec,
                "timestamp": chunk.timestamp_label,
                "source":    chunk.source_file,
            },
        ))

    print(f"Indexing {len(lc_docs)} chunks into '{QDRANT_WHISPER_COLLECTION}' ...")
    for i in range(0, len(lc_docs), batch_size):
        batch = lc_docs[i : i + batch_size]
        try:
            vector_store.add_documents(batch)
        except Exception as e:
            print(f"  ✗ Batch {i//batch_size}: {e}")
        print(f"  [{min(i+batch_size, len(lc_docs))}/{len(lc_docs)}] inserted")

    print(f"✓ Whisper indexing complete: {len(all_ids)} chunks.")
    return all_ids

whisper_ids = index_whisper_chunks(whisper_chunks, whisper_vector_store, whisper_docstore)


Indexing 23 chunks into 'audio_whisper_v1' ...
  [23/23] inserted
✓ Whisper indexing complete: 23 chunks.


### 3.4 BM25 on Transcription


In [8]:
import numpy as np
from rank_bm25 import BM25Okapi
from typing import Tuple

class AudioBM25Index:
    """BM25 index over AudioChunk content."""
    def __init__(self):
        self._chunks: List[AudioChunk] = []
        self._bm25                     = None

    def build(self, chunks: List[AudioChunk]) -> None:
        self._chunks  = chunks
        tokenized     = [c.content.lower().split() for c in chunks]
        self._bm25    = BM25Okapi(tokenized)
        print(f"BM25 built on {len(chunks)} Whisper chunks.")

    def retrieve(self, query: str, top_k: int = RETRIEVER_K) -> List[Tuple[float, AudioChunk]]:
        if self._bm25 is None or not self._chunks:
            return []
        scores  = self._bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self._chunks[i]) for i in top_idx if scores[i] > 0]

whisper_bm25 = AudioBM25Index()
if ENABLE_BM25:
    whisper_bm25.build(whisper_chunks)
else:
    print("BM25 disabled.")


BM25 built on 23 Whisper chunks.


### 3.5 Whisper Hybrid Retrieval


In [9]:
from sentence_transformers import CrossEncoder

# ── Cross-encoder reranker ─────────────────────────────────────────────────
if ENABLE_RERANKING:
    print(f"Loading cross-encoder: {RERANKER_MODEL} ...")
    try:
        reranker           = CrossEncoder(RERANKER_MODEL)
        RERANKER_AVAILABLE = True
        print("  ✓ Reranker ready.")
    except Exception as e:
        print(f"  ✗ {e}")
        reranker           = None
        RERANKER_AVAILABLE = False
else:
    reranker           = None
    RERANKER_AVAILABLE = False
    print("Reranking disabled.")


def whisper_hybrid_retrieve(query: str) -> List[AudioChunk]:
    """
    Hybrid retrieval on the Whisper pipeline:
      1. Dense: text query → text embedding → Qdrant Whisper collection
      2. BM25 : keyword matching on raw transcription chunks
      3. Deduplication by content
      4. Cross-encoder reranking on merged candidates
    """
    seen:       set                     = set()
    candidates: List[Tuple[str, AudioChunk]] = []   # (snippet, chunk)

    # Dense
    try:
        dense_hits = whisper_vector_store.similarity_search(query, k=RETRIEVER_K)
        for lc in dense_hits:
            uid   = lc.metadata.get("doc_id")
            chunk = whisper_docstore.get(uid) if uid else None
            if chunk is None:
                chunk = AudioChunk(content=lc.page_content,
                                   start_sec=lc.metadata.get("start_sec", 0),
                                   end_sec=lc.metadata.get("end_sec", 0))
            if chunk.content not in seen:
                seen.add(chunk.content)
                candidates.append((lc.page_content, chunk))
    except Exception as e:
        print(f"[whisper_retrieve] Dense error: {e}")

    # BM25
    if ENABLE_BM25:
        for score, chunk in whisper_bm25.retrieve(query, top_k=RETRIEVER_K):
            if chunk.content not in seen:
                seen.add(chunk.content)
                candidates.append((chunk.content[:500], chunk))

    if not candidates:
        return []

    # Reranking
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, s) for s, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [c for _, c in candidates]),
                        key=lambda x: x[0], reverse=True)
        final  = [c for _, c in ranked[:RERANKER_TOP_N]]
    else:
        final  = [c for _, c in candidates[:RERANKER_TOP_N]]

    print(f"[whisper_retrieve] {len(candidates)} candidates → {len(final)} "
          f"(query: {query[:55]!r})")
    return final


# Sanity check
test_w = whisper_hybrid_retrieve("What is the attention mechanism?")
print(f"\nRetrieved {len(test_w)} chunks:")
for i, c in enumerate(test_w):
    print(f"  [{i}] {c.timestamp_label}  {c.content[:80]!r}")


Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  ✓ Reranker ready.
[whisper_retrieve] 11 candidates → 4 (query: 'What is the attention mechanism?')

Retrieved 4 chunks:
  [0] [09:10 – 09:37]  'For the pre-training corpus, we use the books corpus and English Wikipedia, extr'
  [1] [02:37 – 03:05]  'The authors use a left-to-right architecture, where every token can only attend '
  [2] [06:48 – 07:18]  'while the GPT transformer uses constrained self-attention where every token can '
  [3] [01:47 – 02:14]  'uses task-specific architectures that include the pre-trained representations as'


---
## 4. CLAP Pipeline — Shared Audio-Text Semantic Space

```
MP3 → split into fixed-length segments (CLAP_SEGMENT_SECS)
    → CLAP audio encoder → audio embeddings
    → Qdrant QDRANT_CLAP_COLLECTION

Query (text) → CLAP text encoder → similarity search → top-k audio segments
    → resolve segment timestamps → extract matching Whisper transcription
    → LLM (text only)
```

The CLAP text encoder and audio encoder share the same vector space,
enabling cross-modal retrieval: a text query finds audio segments by
semantic similarity without requiring keyword overlap in the transcription.


### 4.1 CLAP Model


In [10]:
from transformers import ClapModel, ClapProcessor
import torch
import numpy as np

print(f"Loading CLAP model: {CLAP_MODEL} ...")
clap_device = "cuda" if torch.cuda.is_available() else "cpu"

def clap_output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        return out

    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output

    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state.mean(dim=1)

    if isinstance(out, (tuple, list)):
        return out[0]

    raise TypeError(f"Unsupported CLAP output type: {type(out)}")

try:
    clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL)

    clap_model = ClapModel.from_pretrained(
        CLAP_MODEL,
        torch_dtype=torch.float32,   # <- cambia qui
    ).to(clap_device)

    clap_model.eval()

    with torch.no_grad():
        dummy_audio = np.zeros(SAMPLE_RATE, dtype=np.float32)
        dummy_inputs = clap_processor(
            audio=dummy_audio,
            return_tensors="pt",
            sampling_rate=SAMPLE_RATE,
        )

        dummy_inputs = {k: v.to(clap_device) for k, v in dummy_inputs.items()}

        out = clap_model.get_audio_features(**dummy_inputs)

        audio_emb = clap_output_to_tensor(out)
        audio_emb = torch.nn.functional.normalize(audio_emb, dim=-1)

        CLAP_DIM = audio_emb.shape[-1]

    CLAP_AVAILABLE = True
    print(f"  ✓ CLAP ready on {clap_device} | dim={CLAP_DIM}")

except Exception as e:
    print(f"  ✗ Could not load CLAP: {e}")
    clap_model = clap_processor = None
    CLAP_AVAILABLE = False
    CLAP_DIM = 512


Loading CLAP model: laion/larger_clap_general ...


Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

  ✓ CLAP ready on cuda | dim=512


### 4.2 Audio Segmentation and CLAP Embedding


In [11]:
@dataclass
class AudioSegment:
    """A fixed-length audio segment with timestamp and precomputed CLAP embedding."""
    start_sec:   float
    end_sec:     float
    source_file: str
    transcript:  str = ""    # Whisper text for this time range (populated later)
    doc_type:    str = "text"  # always "text" — LLM receives transcript, not audio

    @property
    def timestamp_label(self) -> str:
        def fmt(s):
            m, sec = divmod(int(s), 60)
            return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"

    @property
    def content(self) -> str:
        """Alias for compatibility with metric functions that expect .content."""
        return self.transcript

def clap_output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "pooler_output"):
        return out.pooler_output
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state.mean(dim=1)
    raise TypeError(f"Unexpected CLAP output type: {type(out)}")

def split_waveform_into_segments(
    waveform:     np.ndarray,
    sr:           int   = SAMPLE_RATE,
    seg_secs:     int   = CLAP_SEGMENT_SECS,
    overlap_secs: int   = CLAP_SEGMENT_OVERLAP,
    source_file:  str   = AUDIO_PATH,
) -> List[AudioSegment]:
    """Split a waveform into overlapping fixed-length AudioSegments."""
    step   = (seg_secs - overlap_secs) * sr
    length = seg_secs * sr
    segs   = []
    pos    = 0
    while pos < len(waveform):
        end_sample = min(pos + length, len(waveform))
        start_s    = pos / sr
        end_s      = end_sample / sr
        segs.append(AudioSegment(start_sec=start_s, end_sec=end_s, source_file=source_file))
        if end_sample == len(waveform):
            break
        pos += step
    return segs


def embed_audio_segments_clap(
    waveform:  np.ndarray,
    segments:  List[AudioSegment],
    batch_size: int = 8,
) -> np.ndarray:
    """
    Embed each AudioSegment with the CLAP audio encoder.
    Returns (N, CLAP_DIM) float32 array, L2-normalised.
    """
    if not CLAP_AVAILABLE:
        return np.zeros((len(segments), CLAP_DIM), dtype=np.float32)

    all_vecs = []
    total    = len(segments)
    print(f"Embedding {total} audio segments with CLAP ...")

    for i in range(0, total, batch_size):
        batch_segs = segments[i : i + batch_size]
        batch_audio = []
        for seg in batch_segs:
            s = int(seg.start_sec * SAMPLE_RATE)
            e = int(seg.end_sec   * SAMPLE_RATE)
            batch_audio.append(waveform[s:e].astype(np.float32))

        try:
            inputs = clap_processor(
                audio=batch_audio,
                return_tensors="pt",
                sampling_rate=SAMPLE_RATE,
                padding=True,
            )

            inputs = {k: v.to(clap_device) for k, v in inputs.items()}

            with torch.no_grad():
                out = clap_model.get_audio_features(**inputs)
                vecs = clap_output_to_tensor(out)
                vecs = torch.nn.functional.normalize(vecs, dim=-1).float()

            all_vecs.append(vecs.cpu().numpy())

        except Exception as e:
            print(f"  ✗ Batch {i//batch_size}: {e} — using zero vectors")
            all_vecs.append(np.zeros((len(batch_segs), CLAP_DIM), dtype=np.float32))

        print(f"  [{min(i+batch_size, total)}/{total}] embedded")

    return np.vstack(all_vecs)


# Segment + embed
clap_segments = split_waveform_into_segments(waveform)
clap_vectors  = embed_audio_segments_clap(waveform, clap_segments)

print(f"\nSegments : {len(clap_segments)}")
print(f"Avg dur  : {sum(s.end_sec-s.start_sec for s in clap_segments)/len(clap_segments):.1f}s")
print(f"Vectors  : {clap_vectors.shape}")


Embedding 79 audio segments with CLAP ...
  [8/79] embedded
  [16/79] embedded
  [24/79] embedded
  [32/79] embedded
  [40/79] embedded
  [48/79] embedded
  [56/79] embedded
  [64/79] embedded
  [72/79] embedded
  [79/79] embedded

Segments : 79
Avg dur  : 9.9s
Vectors  : (79, 512)


### 4.3 Attach Whisper Transcription to CLAP Segments


In [12]:
def get_transcript_for_range(
    segments: List[dict],
    start_s:  float,
    end_s:    float,
    padding:  float = 1.0,
) -> str:
    """
    Extract the Whisper transcription covering [start_s - padding, end_s + padding].
    Uses the segment-level timestamps already available from the transcription.
    """
    relevant = []
    for seg in segments:
        ts_start = seg["timestamp"][0] or 0.0
        ts_end   = seg["timestamp"][1] or AUDIO_DURATION_SECS
        # Include segment if it overlaps with the query window
        if ts_end >= (start_s - padding) and ts_start <= (end_s + padding):
            relevant.append(seg["text"])
    return " ".join(relevant).strip()


# Attach transcript to every CLAP segment
print("Attaching Whisper transcription to CLAP segments ...")
for seg in clap_segments:
    seg.transcript = get_transcript_for_range(segments, seg.start_sec, seg.end_sec)

# How many segments have non-empty transcription?
with_text = sum(1 for s in clap_segments if s.transcript.strip())
print(f"✓ {with_text}/{len(clap_segments)} segments have transcription coverage.")

# Preview
print(f"\nFirst CLAP segment {clap_segments[0].timestamp_label}:")
print(f"  Transcript: {clap_segments[0].transcript[:200]!r}")


Attaching Whisper transcription to CLAP segments ...
✓ 79/79 segments have transcription coverage.

First CLAP segment [00:00 – 00:10]:
  Transcript: 'BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language'


### 4.4 CLAP Qdrant Collection + Indexing


In [13]:
from qdrant_client.http.models import PointStruct

# ── CLAP collection ───────────────────────────────────────────────────────────
if RESET_CLAP_COLLECTION and client.collection_exists(QDRANT_CLAP_COLLECTION):
    client.delete_collection(QDRANT_CLAP_COLLECTION)
    print(f"✓ Deleted '{QDRANT_CLAP_COLLECTION}'")

if not client.collection_exists(QDRANT_CLAP_COLLECTION):
    client.create_collection(
        collection_name = QDRANT_CLAP_COLLECTION,
        vectors_config  = VectorParams(size=CLAP_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created CLAP collection '{QDRANT_CLAP_COLLECTION}' (dim={CLAP_DIM})")
else:
    print(f"✓ Using existing CLAP collection '{QDRANT_CLAP_COLLECTION}'")

# Docstore: UUID → AudioSegment
clap_docstore: Dict[str, AudioSegment] = {}

# ── Index all segments ────────────────────────────────────────────────────────
UPSERT_BATCH = 64
clap_ids     = []
points       = []

for seg, vec in zip(clap_segments, clap_vectors):
    uid = str(uuid.uuid4())
    clap_ids.append(uid)
    clap_docstore[uid] = seg
    points.append(PointStruct(
        id      = uid,
        vector  = vec.tolist(),
        payload = {
            "start_sec": seg.start_sec,
            "end_sec":   seg.end_sec,
            "timestamp": seg.timestamp_label,
            "source":    seg.source_file,
            "preview":   seg.transcript[:100],
        },
    ))

print(f"Inserting {len(points)} CLAP vectors into '{QDRANT_CLAP_COLLECTION}' ...")
inserted = 0
for i in range(0, len(points), UPSERT_BATCH):
    batch = points[i : i + UPSERT_BATCH]
    try:
        client.upsert(collection_name=QDRANT_CLAP_COLLECTION, points=batch)
        inserted += len(batch)
    except Exception as e:
        print(f"  ✗ Batch {i//UPSERT_BATCH}: {e}")

print(f"✓ CLAP indexing complete: {inserted}/{len(points)} segments.")


✓ Created CLAP collection 'audio_clap_v1' (dim=512)
Inserting 79 CLAP vectors into 'audio_clap_v1' ...
✓ CLAP indexing complete: 79/79 segments.


### 4.5 CLAP Hybrid Retrieval


In [14]:
def encode_query_clap(query: str) -> np.ndarray:
    """
    Encode a text query with the CLAP text encoder.
    Returns a normalised float32 vector of shape (CLAP_DIM,).
    The CLAP text and audio encoders share the same space:
    text queries can directly retrieve audio segments by cosine similarity.
    """
    if not CLAP_AVAILABLE:
        raise RuntimeError("CLAP model not available.")
    inputs = clap_processor(text=[query], return_tensors="pt", padding=True).to(clap_device)
    with torch.no_grad():
        vec = clap_model.get_text_features(**inputs)
        vec = clap_output_to_tensor(vec)
        vec = torch.nn.functional.normalize(vec, dim=-1).float()
    return vec.cpu().numpy()[0]


def clap_retrieve(query: str) -> List[AudioSegment]:
    """
    CLAP retrieval:
      1. Encode query with CLAP text encoder → query vector
      2. Cosine similarity search in QDRANT_CLAP_COLLECTION → audio segments
      3. Cross-encoder reranking on the *transcript* text of retrieved segments
         (CLAP retrieves by audio similarity; reranker refines by textual relevance)

    Note: BM25 is intentionally not used in this pipeline.
    CLAP embeddings already capture semantic audio content; keyword
    matching on the transcription would mix two retrieval signals
    that operate at different abstraction levels.
    """
    if not CLAP_AVAILABLE:
        print("[clap_retrieve] CLAP not available.")
        return []

    seen:        set                         = set()
    candidates:  List[Tuple[str, AudioSegment]] = []

    try:
        query_vec = encode_query_clap(query)
        result = client.query_points(
            collection_name = QDRANT_CLAP_COLLECTION,
            query           = query_vec.tolist(),
            limit           = RETRIEVER_K,
            with_payload    = True,
        )

        hits = result.points

        for h in hits:
            seg = clap_docstore.get(str(h.id))
            if seg is None or seg.content in seen:
                continue
            seen.add(seg.content)
            candidates.append((seg.transcript[:500], seg))
    except Exception as e:
        print(f"[clap_retrieve] Error: {e}")
        return []

    if not candidates:
        return []

    # Rerank on transcript text
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, t) for t, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [s for _, s in candidates]),
                        key=lambda x: x[0], reverse=True)
        final  = [s for _, s in ranked[:RERANKER_TOP_N]]
    else:
        final  = [s for _, s in candidates[:RERANKER_TOP_N]]

    print(f"[clap_retrieve] {len(hits)} hits → {len(final)} after rerank "
          f"(query: {query[:55]!r})")
    return final


# Sanity check
test_c = clap_retrieve("What is the attention mechanism?")
print(f"\nRetrieved {len(test_c)} segments:")
for i, s in enumerate(test_c):
    print(f"  [{i}] {s.timestamp_label}  {s.transcript[:80]!r}")


[clap_retrieve] 8 hits → 4 after rerank (query: 'What is the attention mechanism?')

Retrieved 4 segments:
  [0] [02:40 – 02:50]  'The authors use a left-to-right architecture, where every token can only attend '
  [1] [00:40 – 00:50]  'BERT is designed to pre-trained deep bi-directional representations from unlabel'
  [2] [03:28 – 03:38]  'Byrd alleviates the previously mentioned unidirectionality constraint by using a'
  [3] [06:08 – 06:18]  'During pre-training, the model is trained on unlabeled data over different pre-t'


---
## 5. Generation Model

Both pipelines pass **text only** to the LLM:
the Whisper pipeline passes transcribed chunk text,
the CLAP pipeline passes the Whisper transcript of the retrieved audio segment.


In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR_AVAILABLE     = False
USE_OLLAMA_GENERATOR    = False
gen_tokenizer           = None
gen_text_model          = None

if GENERATION_BACKEND == "ollama":
    try:
        from langchain_ollama import ChatOllama
        ollama_llm          = ChatOllama(model=GENERATION_MODEL, temperature=0)
        USE_OLLAMA_GENERATOR = True
        GENERATOR_AVAILABLE  = True
        print(f"✓ Ollama generator ready: {GENERATION_MODEL}")
    except Exception as e:
        print(f"✗ Ollama: {e}")
else:
    print(f"Loading HF generation model: {GENERATION_MODEL} ...")
    try:
        gen_tokenizer  = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=True)
        gen_text_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL,
            torch_dtype = HF_TORCH_DTYPE,
            device_map  = HF_DEVICE_MAP,
            trust_remote_code = True,
        )
        gen_text_model.eval()
        GENERATOR_AVAILABLE = True
        print(f"  ✓ Generator ready: {GENERATION_MODEL}")
    except Exception as e:
        print(f"  ✗ {e}")


def generate_answer(retrieved: List, question: str) -> str:
    """
    Build a text-only context from retrieved AudioChunk / AudioSegment objects
    and generate an answer with the local model.
    Always text-only: audio content is represented via its Whisper transcript.
    """
    if not GENERATOR_AVAILABLE:
        return "[Generator unavailable]"

    context_parts = []
    for item in retrieved:
        ts   = getattr(item, "timestamp_label", "")
        text = item.content.strip()
        if text:
            context_parts.append(f"{ts}\n{text}")

    context_str = "\n\n".join(context_parts) if context_parts else "[No context retrieved]"
    prompt = (
        "Answer the question using only the provided context from an audio transcript. "
        "If the context is insufficient, say so explicitly.\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {question}"
    )

    if USE_OLLAMA_GENERATOR:
        return ollama_llm.invoke(prompt).content.strip()

    messages = [{"role": "user", "content": prompt}]
    if hasattr(gen_tokenizer, "apply_chat_template"):
        text_in = gen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    else:
        text_in = prompt

    inputs = gen_tokenizer([text_in], return_tensors="pt").to(gen_text_model.device)
    with torch.no_grad():
        generated = gen_text_model.generate(
            **inputs,
            max_new_tokens = GENERATION_MAX_NEW_TOKENS,
            do_sample      = False,
            temperature    = None,
            pad_token_id   = gen_tokenizer.eos_token_id,
        )
    new_tokens = generated[:, inputs.input_ids.shape[1]:]
    return gen_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


Loading HF generation model: Qwen/Qwen2.5-3B-Instruct ...
  ✗ CUDA out of memory. Tried to allocate 4.54 GiB. GPU 0 has a total capacity of 11.99 GiB of which 5.68 GiB is free. Of the allocated memory 2.94 GiB is allocated by PyTorch, and 26.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


In [16]:
# Quick generation test — both pipelines
q = "How is BERT positioned in regard to transformers?"

print("── Whisper pipeline ─────────────────────────────────────────────────")
w_docs = whisper_hybrid_retrieve(q)
w_answer = generate_answer(w_docs, q)
print(w_answer)

print("\n── CLAP pipeline ────────────────────────────────────────────────────")
c_docs = clap_retrieve(q)
c_answer = generate_answer(c_docs, q)
print(c_answer)


── Whisper pipeline ─────────────────────────────────────────────────
[whisper_retrieve] 11 candidates → 4 (query: 'How is BERT positioned in regard to transformers?')
[Generator unavailable]

── CLAP pipeline ────────────────────────────────────────────────────
[clap_retrieve] 8 hits → 4 after rerank (query: 'How is BERT positioned in regard to transformers?')
[Generator unavailable]


---
## 6. Evaluation

This section evaluates the audio RAG approaches with answer-level and retrieval-level metrics:

| Metric | What it checks |
|---|---|
| `BERTScore` | semantic similarity between generated and expected answer |
| `Precision` / `Recall` | lexical overlap of answer tokens against the expected answer |
| `Context Recall` | how many annotated claims are present in retrieved transcript context |
| `Timestamp Coverage` | how many inferred evidence time windows are covered by retrieved audio chunks/segments |
| `Must` / `Should` recall | claim coverage split by required vs desirable facts |

Run `6.4 Generate Evaluation Answers` once to cache retrieval, answers, and metrics. The widget after it only filters cached rows, so changing selectors does not regenerate answers.


### 6.1 Test Questions and Annotated Claims


In [17]:
TEST_SET = [
    {
        "question": "Describe the overall architecture of the Transformer model",
        
        "expected_answer":
            "The Transformer uses an encoder-decoder architecture. The encoder consists "
            "of 6 identical layers, each with multi-head self-attention and position-wise "
            "feed-forward networks. The decoder also has 6 layers with an additional "
            "masked self-attention mechanism and encoder-decoder attention. Residual "
            "connections and layer normalization are applied around each sub-layer.",
        
        "claims": [
            {
                "text": "The Transformer has an encoder-decoder architecture",
                "importance": "must",
                "source": "audio"  # spoken architecture explanation
            },
            {
                "text": "The encoder has 6 identical layers",
                "importance": "must",
                "source": "audio"
            },
            {
                "text": "Each encoder layer contains multi-head self-attention and feed-forward networks",
                "importance": "must",
                "source": "audio"
            },
            {
                "text": "The decoder also has 6 layers",
                "importance": "should",
                "source": "audio"
            },
            {
                "text": "The decoder includes masked self-attention and encoder-decoder attention",
                "importance": "should",
                "source": "audio"
            },
            {
                "text": "Residual connections and layer normalization are used",
                "importance": "should",
                "source": "audio"
            },
        ],
        
        "requires_continuity": False,
        "requires_timestamp": True,
        
        # Legacy keywords (naive testing approach - kept for reference)
        "context_keywords": ["encoder", "decoder", "layers", "self-attention", "feed-forward"],
        "answer_keywords": ["encoder", "decoder", "6 layers", "attention", "residual"],
    },
    

    {
        "question": "What BLEU scores did the Transformer achieve on WMT 2014 English-to-German translation?",
        
        "expected_answer":
            "The Transformer (big) model achieved 28.4 BLEU on the WMT 2014 English-to-German "
            "translation task, outperforming previous state-of-the-art models including ensembles.",
        
        "claims": [
            {
                "text": "The model achieved 28.4 BLEU on WMT 2014 EN-DE",
                "importance": "must",
                "source": "audio"  # spoken results mention
            },
            {
                "text": "This outperformed previous state-of-the-art",
                "importance": "should",
                "source": "audio"  # spoken discussion
            },
            {
                "text": "The big model configuration was used",
                "importance": "should",
                "source": "audio"  # spoken model/configuration mention
            },
        ],
        
        "requires_continuity": False,
        "requires_timestamp": True,
        
        # Legacy keywords
        "context_keywords": ["BLEU", "28.4", "WMT", "2014", "English", "German"],
        "answer_keywords": ["28.4", "BLEU", "WMT 2014"],
    },
    

    {
        "question": "What is the formula for scaled dot-product attention?",
        
        "expected_answer":
            "The scaled dot-product attention is computed as Attention(Q, K, V) = "
            "softmax(QK^T / sqrt(d_k)) V, where Q are queries, K are keys, V are values, "
            "and d_k is the dimension of the keys.",
        
        "claims": [
            {
                "text": "The formula is softmax of QK transpose divided by square root of d_k, times V",
                "importance": "must",
                "source": "audio"  # spoken or quoted formula
            },
            {
                "text": "Q represents queries, K represents keys, V represents values",
                "importance": "must",
                "source": "audio"
            },
            {
                "text": "d_k is the dimension of the keys",
                "importance": "should",
                "source": "text"
            },
            {
                "text": "The scaling factor prevents gradients from becoming too small",
                "importance": "should",
                "source": "text"
            },
        ],
        
        "requires_continuity": False,
        "requires_timestamp": True,
        
        # Legacy keywords
        "context_keywords": ["softmax", "dot product", "queries", "keys", "values", "sqrt", "d_k"],
        "answer_keywords": ["softmax", "QK", "sqrt", "d_k", "V"],
    },
    

    {
        "question": "What is positional encoding and why is it necessary in the Transformer?",
        
        "expected_answer":
            "Positional encodings are added to the input embeddings to inject information about "
            "the relative or absolute position of tokens in the sequence. They are necessary "
            "because the Transformer contains no recurrence or convolution, so it has no inherent "
            "notion of token order. The paper uses sine and cosine functions of different frequencies.",
        
        "claims": [
            {
                "text": "Positional encodings are added to input embeddings",
                "importance": "must",
                "source": "text"
            },
            {
                "text": "They provide information about token position in the sequence",
                "importance": "must",
                "source": "text"
            },
            {
                "text": "The Transformer has no recurrence or convolution",
                "importance": "must",
                "source": "text"
            },
            {
                "text": "Therefore it needs explicit position information",
                "importance": "should",
                "source": "text"
            },
            {
                "text": "Sine and cosine functions of different frequencies are used",
                "importance": "should",
                "source": "audio"  # spoken positional-encoding detail
            },
        ],
        
        "requires_continuity": True,   # spans motivation + solution
        "requires_timestamp": True,        # timestamped audio evidence helps checking
        
        # Legacy keywords
        "context_keywords": ["positional", "encoding", "position", "sequence", "sine", "cosine"],
        "answer_keywords": ["position", "encoding", "sequence", "order"],
    },
    

    {
        "question": "What optimizer was used to train the Transformer?",
        
        "expected_answer":
            "The Transformer was trained using the Adam optimizer with beta1=0.9, beta2=0.98, "
            "and epsilon=1e-9. The learning rate was varied during training using a warmup "
            "schedule that increases linearly for the first warmup steps, then decreases "
            "proportionally to the inverse square root of the step number.",
        
        "claims": [
            {
                "text": "The Adam optimizer was used",
                "importance": "must",
                "source": "text"
            },
            {
                "text": "Beta1=0.9 and beta2=0.98",
                "importance": "should",
                "source": "text"
            },
            {
                "text": "A warmup learning rate schedule was used",
                "importance": "should",
                "source": "text"
            },
            {
                "text": "Learning rate increases linearly during warmup then decays",
                "importance": "should",
                "source": "text"
            },
        ],
        
        "requires_continuity": False,
        "requires_timestamp": True,
        
        # Legacy keywords
        "context_keywords": ["adam", "optimizer", "beta", "warmup", "learning rate"],
        "answer_keywords": ["adam", "warmup"],
    },

    {
        "question": "How does the decoder differ from the encoder in the Transformer?",
        
        "expected_answer":
            "The decoder has an additional masked multi-head self-attention layer that prevents "
            "positions from attending to subsequent positions, preserving the autoregressive "
            "property. It also includes an encoder-decoder attention sub-layer that attends "
            "over the encoder's output, whereas the encoder only has self-attention and "
            "feed-forward sub-layers.",
        
        "claims": [
            {
                "text": "The decoder has masked self-attention",
                "importance": "must",
                "source": "audio"  # spoken decoder explanation
            },
            {
                "text": "Masking prevents attending to future positions",
                "importance": "must",
                "source": "text"
            },
            {
                "text": "The decoder includes encoder-decoder attention",
                "importance": "must",
                "source": "audio"
            },
            {
                "text": "This preserves the autoregressive property",
                "importance": "should",
                "source": "text"
            },
            {
                "text": "The encoder only has self-attention and feed-forward layers",
                "importance": "should",
                "source": "audio"
            },
        ],
        
        "requires_continuity": True,    # needs to discuss both components
        "requires_timestamp": True,        # timestamped audio evidence is available
        
        # Legacy keywords
        "context_keywords": ["encoder", "decoder", "masked", "self-attention", "cross-attention"],
        "answer_keywords": ["decoder", "masked", "encoder-decoder attention"],
    },
    
    {
        "question": "Explain how multi-head attention works",
        
        "expected_answer":
            "Multi-head attention performs multiple attention operations in parallel. "
            "Instead of performing a single attention function, queries, keys and values "
            "are linearly projected h times with different learned projections. Attention "
            "is performed on each of these projected versions in parallel, then the outputs "
            "are concatenated and projected again to produce the final values.",
        
        "claims": [
            {
                "text": "Multi-head attention performs multiple attention operations in parallel",
                "importance": "must",
                "source": "audio"
            },
            {
                "text": "Queries, keys, and values are linearly projected h times",
                "importance": "must",
                "source": "audio"  # spoken explanation of heads
            },
            {
                "text": "Each projection uses different learned parameters",
                "importance": "should",
                "source": "text"
            },
            {
                "text": "Attention is performed on each projected version",
                "importance": "must",
                "source": "audio"  # spoken explanation of parallel heads
            },
            {
                "text": "The outputs are concatenated and linearly projected",
                "importance": "should",
                "source": "audio"
            },
        ],
        
        "requires_continuity": False,
        "requires_timestamp": True,         # timestamped audio evidence for parallel heads
        
        # Legacy keywords
        "context_keywords": ["multi-head", "attention", "parallel", "projection", "concatenate"],
        "answer_keywords": ["multi-head", "parallel", "projection", "heads"],
    },
]


for item in TEST_SET:
    item.setdefault("timestamp_targets", [])

print(f"Evaluation test set: {len(TEST_SET)} questions.")
print(f"  Timestamp questions: {sum(1 for q in TEST_SET if q.get('requires_timestamp', True))}")
print(f"  Must claims        : {sum(1 for q in TEST_SET for c in q['claims'] if c['importance'] == 'must')}")
print(f"  Should claims      : {sum(1 for q in TEST_SET for c in q['claims'] if c['importance'] == 'should')}")


Evaluation test set: 7 questions.
  Timestamp questions: 7
  Must claims        : 16
  Should claims      : 16


### 6.2 Retrieval Views and Audio RAG Approach Registry


In [18]:
import math
import re
from collections import defaultdict
from html import escape
from typing import Any, Callable, Iterable

try:
    import pandas as pd
except Exception as e:
    pd = None
    print(f"pandas unavailable; tables will be shown as raw lists. Error: {e}")

try:
    from IPython.display import HTML, clear_output, display
except Exception:
    HTML = None
    clear_output = None

from bert_score import score as bert_score_score

BERTSCORE_MODEL_TYPE = os.getenv("BERTSCORE_MODEL_TYPE", "distilbert-base-uncased")
CLAIM_MATCH_THRESHOLD = float(os.getenv("CLAIM_MATCH_THRESHOLD", "0.45"))
TIMESTAMP_COVERAGE_MIN_OVERLAP = float(os.getenv("TIMESTAMP_COVERAGE_MIN_OVERLAP", "0.30"))
TIMESTAMP_EVIDENCE_MIN_SCORE = float(os.getenv("TIMESTAMP_EVIDENCE_MIN_SCORE", "0.35"))
RETRIEVAL_STAGE_OPTIONS = {
    "dense": "Dense retrieval only",
    "bm25": "BM25 only",
    "hybrid": "Dense + BM25",
    "rerank": "Cross-encoder rerank",
}
APPROACH_RETRIEVAL_STAGES = {
    "whisper": ["dense", "bm25", "hybrid", "rerank"],
    "clap": ["dense", "rerank"],
}
APPROACH_STAGE_LABELS = {
    "whisper": {
        "dense": "Whisper dense retrieval only",
        "bm25": "Whisper BM25 only",
        "hybrid": "Whisper dense + BM25",
        "rerank": "Whisper dense + BM25 + cross-encoder rerank",
    },
    "clap": {
        "dense": "CLAP audio-text dense retrieval only",
        "rerank": "CLAP dense + transcript cross-encoder rerank",
    },
}


def retrieval_stage_label(approach_key: str, stage: str) -> str:
    return APPROACH_STAGE_LABELS.get(approach_key, {}).get(
        stage,
        RETRIEVAL_STAGE_OPTIONS.get(stage, stage),
    )

_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "because", "by", "for", "from",
    "has", "have", "in", "into", "is", "it", "its", "of", "on", "or", "over",
    "that", "the", "their", "then", "there", "this", "to", "used", "using", "was",
    "were", "with", "where", "which", "while",
}


def _normalize_text(text: str) -> str:
    text = str(text or "").lower()
    text = text.replace("^", " ").replace("_", "_")
    text = re.sub(r"[^a-z0-9_\.]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _tokens(text: str) -> list[str]:
    return [t for t in _normalize_text(text).split() if len(t) > 1 and t not in _STOPWORDS]


def answer_precision_recall(candidate: str, reference: str) -> dict[str, float]:
    cand = set(_tokens(candidate))
    ref = set(_tokens(reference))
    if not cand and not ref:
        return {"precision": 1.0, "recall": 1.0, "answer_f1": 1.0}
    if not cand or not ref:
        return {"precision": 0.0, "recall": 0.0, "answer_f1": 0.0}
    overlap = len(cand & ref)
    precision = overlap / len(cand)
    recall = overlap / len(ref)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "answer_f1": f1}


def claim_match_score(claim_text: str, target_text: str) -> float:
    claim_tokens = set(_tokens(claim_text))
    target_tokens = set(_tokens(target_text))
    if not claim_tokens:
        return 0.0
    if _normalize_text(claim_text) and _normalize_text(claim_text) in _normalize_text(target_text):
        return 1.0
    token_recall = len(claim_tokens & target_tokens) / len(claim_tokens)
    token_precision = len(claim_tokens & target_tokens) / len(target_tokens) if target_tokens else 0.0
    token_f1 = 2 * token_precision * token_recall / (token_precision + token_recall) if token_precision + token_recall else 0.0
    return max(token_recall, token_f1)


def _filtered_claims(item: dict, importance: str | None = None) -> list[dict]:
    claims = item.get("claims", [])
    if importance is not None:
        claims = [c for c in claims if c.get("importance") == importance]
    return claims


def claim_recall(item: dict, target_text: str, importance: str | None = None) -> tuple[float, int, int]:
    claims = _filtered_claims(item, importance=importance)
    if not claims:
        return (math.nan, 0, 0)
    covered = sum(1 for c in claims if claim_match_score(c["text"], target_text) >= CLAIM_MATCH_THRESHOLD)
    return covered / len(claims), covered, len(claims)


def retrieved_context_text(docs: list[Any]) -> str:
    parts = []
    for i, doc in enumerate(docs, start=1):
        ts = getattr(doc, "timestamp_label", "")
        text = getattr(doc, "content", "")
        start = getattr(doc, "start_sec", None)
        end = getattr(doc, "end_sec", None)
        interval = f" [{start:.1f}-{end:.1f}s]" if start is not None and end is not None else ""
        parts.append(f"[retrieved {i}{interval}] {ts}\n{text}")
    return "\n\n".join(parts)


def _dedupe_key(doc: Any) -> tuple:
    return (
        round(float(getattr(doc, "start_sec", 0.0)), 2),
        round(float(getattr(doc, "end_sec", 0.0)), 2),
        hash(getattr(doc, "content", "")),
    )


def _append_candidate(candidates: list[tuple[str, Any]], seen: set, rerank_text: str, doc: Any) -> None:
    key = _dedupe_key(doc)
    if key not in seen:
        seen.add(key)
        candidates.append((rerank_text or getattr(doc, "content", "")[:500], doc))


def _interval(doc: Any) -> tuple[float, float] | None:
    start = getattr(doc, "start_sec", None)
    end = getattr(doc, "end_sec", None)
    if start is None or end is None:
        return None
    start = float(start)
    end = float(end)
    if end <= start:
        return None
    return (start, end)


def _overlap_seconds(a: tuple[float, float], b: tuple[float, float]) -> float:
    return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))


def _merge_intervals(intervals: list[tuple[float, float]]) -> list[tuple[float, float]]:
    if not intervals:
        return []
    ordered = sorted(intervals)
    merged = [ordered[0]]
    for start, end in ordered[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


def _target_covered(target: tuple[float, float], retrieved_intervals: list[tuple[float, float]]) -> bool:
    duration = max(1e-9, target[1] - target[0])
    covered = sum(_overlap_seconds(target, interval) for interval in _merge_intervals(retrieved_intervals))
    return (covered / duration) >= TIMESTAMP_COVERAGE_MIN_OVERLAP


def infer_timestamp_targets(item: dict, top_k_per_claim: int = 1) -> list[dict[str, Any]]:
    """Infer gold-ish evidence windows by matching each claim against Whisper chunks."""
    if item.get("timestamp_targets"):
        return item["timestamp_targets"]

    targets = []
    for claim in item.get("claims", []):
        scored = [
            (claim_match_score(claim["text"], chunk.content), chunk)
            for chunk in whisper_chunks
            if getattr(chunk, "content", "").strip()
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        for score, chunk in scored[:top_k_per_claim]:
            if score >= TIMESTAMP_EVIDENCE_MIN_SCORE:
                targets.append({
                    "claim": claim["text"],
                    "start_sec": float(chunk.start_sec),
                    "end_sec": float(chunk.end_sec),
                    "score": float(score),
                    "timestamp": chunk.timestamp_label,
                })
    return targets


def timestamp_coverage(item: dict, retrieved_docs: list[Any]) -> tuple[float, int, int, str]:
    targets = infer_timestamp_targets(item)
    if not targets:
        return (math.nan, 0, 0, "")
    retrieved_intervals = [interval for interval in (_interval(doc) for doc in retrieved_docs) if interval is not None]
    covered = 0
    labels = []
    for target in targets:
        target_interval = (float(target["start_sec"]), float(target["end_sec"]))
        hit = _target_covered(target_interval, retrieved_intervals)
        covered += int(hit)
        labels.append(f"{target.get('timestamp', f'{target_interval[0]:.1f}-{target_interval[1]:.1f}s')}:{'hit' if hit else 'miss'}")
    return covered / len(targets), covered, len(targets), "; ".join(labels)


def retrieve_whisper_by_stage(query: str, stage: str = "rerank", k: int = RETRIEVER_K, top_n: int = RERANKER_TOP_N) -> list[AudioChunk]:
    stage = stage if stage in RETRIEVAL_STAGE_OPTIONS else "rerank"
    seen: set = set()
    candidates: list[tuple[str, AudioChunk]] = []

    if stage in ("dense", "hybrid", "rerank"):
        try:
            for lc in whisper_vector_store.similarity_search(query, k=k):
                uid = lc.metadata.get("doc_id")
                chunk = whisper_docstore.get(uid) if uid else None
                if chunk is None:
                    chunk = AudioChunk(
                        content=lc.page_content,
                        start_sec=lc.metadata.get("start_sec", 0),
                        end_sec=lc.metadata.get("end_sec", 0),
                        source_file=lc.metadata.get("source", AUDIO_PATH),
                    )
                _append_candidate(candidates, seen, lc.page_content, chunk)
        except Exception as e:
            print(f"[retrieve_whisper_by_stage] Dense retrieval error: {e}")

    if stage in ("bm25", "hybrid", "rerank"):
        for score, chunk in whisper_bm25.retrieve(query, top_k=k):
            _append_candidate(candidates, seen, chunk.content[:500], chunk)

    if not candidates:
        return []
    if stage == "rerank" and RERANKER_AVAILABLE and reranker is not None:
        pairs = [(query, text) for text, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [d for _, d in candidates]), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:top_n]]
    return [d for _, d in candidates[:top_n]]


def retrieve_clap_by_stage(query: str, stage: str = "rerank", k: int = RETRIEVER_K, top_n: int = RERANKER_TOP_N) -> list[AudioSegment]:
    """
    Parameterized CLAP retrieval for evaluation.

    Valid CLAP views:
      - dense: text query in CLAP shared space retrieves audio segments
      - rerank: same CLAP dense candidates, then transcript cross-encoder reranking

    BM25 is intentionally not exposed here: it operates on transcripts, not on
    CLAP's audio-text embedding space, so it would be a different retrieval
    baseline rather than a CLAP retrieval mode.
    """
    if stage not in APPROACH_RETRIEVAL_STAGES["clap"]:
        print(f"[retrieve_clap_by_stage] Retrieval stage {stage!r} is not applicable to CLAP.")
        return []
    if not CLAP_AVAILABLE:
        print("[retrieve_clap_by_stage] CLAP not available.")
        return []

    seen: set = set()
    candidates: list[tuple[str, AudioSegment]] = []

    try:
        query_vec = encode_query_clap(query)
        result = client.query_points(
            collection_name=QDRANT_CLAP_COLLECTION,
            query=query_vec.tolist(),
            limit=k,
            with_payload=True,
        )
        for hit in result.points:
            seg = clap_docstore.get(str(hit.id))
            if seg is not None:
                _append_candidate(candidates, seen, seg.content[:500], seg)
    except Exception as e:
        print(f"[retrieve_clap_by_stage] CLAP dense retrieval error: {e}")

    if not candidates:
        return []
    if stage == "rerank" and RERANKER_AVAILABLE and reranker is not None:
        pairs = [(query, text) for text, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [d for _, d in candidates]), key=lambda x: x[0], reverse=True)
        return [d for _, d in ranked[:top_n]]
    return [d for _, d in candidates[:top_n]]


RAG_APPROACHES = {
    "whisper": {
        "label": "Whisper transcript RAG",
        "retriever": retrieve_whisper_by_stage,
        "description": "Dense/BM25/rerank over Whisper transcript chunks.",
    },
    "clap": {
        "label": "CLAP audio-text RAG",
        "retriever": retrieve_clap_by_stage,
        "description": "CLAP audio-text dense retrieval, optionally reranked with a text cross-encoder over attached transcripts.",
    },
}

print("Audio evaluation harness ready.")
print("Retrieval views by approach:")
for approach_key, stages in APPROACH_RETRIEVAL_STAGES.items():
    labels = ", ".join(retrieval_stage_label(approach_key, stage) for stage in stages)
    print(f"  {RAG_APPROACHES[approach_key]['label']}: {labels}")


Audio evaluation harness ready.
Retrieval views by approach:
  Whisper transcript RAG: Whisper dense retrieval only, Whisper BM25 only, Whisper dense + BM25, Whisper dense + BM25 + cross-encoder rerank
  CLAP audio-text RAG: CLAP audio-text dense retrieval only, CLAP dense + transcript cross-encoder rerank


### 6.3 Metrics


In [19]:
def compute_bertscore_batch(candidates: list[str], references: list[str]) -> list[dict[str, float]]:
    """Compute BERTScore with the installed `bert_score` package."""
    if not candidates:
        return []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    p, r, f1 = bert_score_score(
        candidates,
        references,
        lang="en",
        model_type=BERTSCORE_MODEL_TYPE,
        verbose=False,
        rescale_with_baseline=False,
        device=device,
    )
    return [
        {
            "bertscore_precision": float(pp),
            "bertscore_recall": float(rr),
            "bertscore_f1": float(ff),
            "bertscore_backend": "bert_score",
        }
        for pp, rr, ff in zip(p, r, f1)
    ]


def evaluate_single_result(item: dict, answer: str, retrieved_docs: list[Any]) -> dict[str, Any]:
    context_text = retrieved_context_text(retrieved_docs)
    lexical = answer_precision_recall(answer, item["expected_answer"])

    must_recall, must_hit, must_total = claim_recall(item, answer, importance="must")
    should_recall, should_hit, should_total = claim_recall(item, answer, importance="should")
    answer_claim_recall, answer_claim_hit, answer_claim_total = claim_recall(item, answer)
    context_recall, context_hit, context_total = claim_recall(item, context_text)
    ts_coverage, ts_hit, ts_total, ts_detail = timestamp_coverage(item, retrieved_docs)

    return {
        **lexical,
        "context_recall": context_recall,
        "timestamp_coverage": ts_coverage,
        "must_recall": must_recall,
        "should_recall": should_recall,
        "answer_claim_recall": answer_claim_recall,
        "must_covered": f"{must_hit}/{must_total}",
        "should_covered": f"{should_hit}/{should_total}",
        "context_claims_covered": f"{context_hit}/{context_total}",
        "timestamp_targets_covered": f"{ts_hit}/{ts_total}",
        "timestamp_detail": ts_detail,
        "n_retrieved": len(retrieved_docs),
        "retrieved_seconds": sum(max(0.0, float(getattr(d, "end_sec", 0)) - float(getattr(d, "start_sec", 0))) for d in retrieved_docs),
    }


def _build_eval_row(
    item: dict,
    q_idx: int,
    approach_key: str,
    stage: str,
    compute_bert_later: bool = True,
) -> tuple[dict[str, Any], str | None, str | None]:
    spec = RAG_APPROACHES[approach_key]
    stage_label = retrieval_stage_label(approach_key, stage)
    row = {
        "approach_key": approach_key,
        "approach": spec["label"],
        "retrieval_view": stage_label,
        "retrieval_stage": stage,
        "question_id": q_idx,
        "question": item["question"],
        "requires_timestamp": item.get("requires_timestamp", True),
    }
    try:
        retrieved = spec["retriever"](item["question"], stage=stage)
        answer = generate_answer(retrieved, item["question"])
        row.update(evaluate_single_result(item, answer, retrieved))
        row["answer"] = answer
        row["source_preview"] = retrieved_context_text(retrieved)[:1000]
        if compute_bert_later:
            return row, answer, item["expected_answer"]
    except Exception as e:
        row.update({
            "error": repr(e),
            "precision": math.nan,
            "recall": math.nan,
            "answer_f1": math.nan,
            "context_recall": math.nan,
            "timestamp_coverage": math.nan,
            "must_recall": math.nan,
            "should_recall": math.nan,
            "answer_claim_recall": math.nan,
            "n_retrieved": 0,
            "retrieved_seconds": 0.0,
            "answer": "",
            "source_preview": "",
            "timestamp_targets_covered": "0/0",
            "timestamp_detail": "",
        })
    return row, None, None


def _stages_for_approach(
    approach_key: str,
    stages: Iterable[str] | None = None,
    stages_by_approach: dict[str, Iterable[str]] | None = None,
) -> list[str]:
    valid = APPROACH_RETRIEVAL_STAGES.get(approach_key, list(RETRIEVAL_STAGE_OPTIONS))
    requested = list(stages_by_approach.get(approach_key, valid)) if stages_by_approach else list(stages or valid)
    return [stage for stage in requested if stage in valid]


def generate_evaluation_results(
    test_set: list[dict] = TEST_SET,
    approaches: Iterable[str] | None = None,
    stages: Iterable[str] | None = None,
    stages_by_approach: dict[str, Iterable[str]] | None = None,
    limit: int | None = None,
    compute_bert: bool = True,
) -> Any:
    approaches = list(approaches or RAG_APPROACHES.keys())
    selected_questions = test_set[:limit] if limit else test_set
    approach_stage_pairs = [
        (approach_key, stage)
        for approach_key in approaches
        for stage in _stages_for_approach(approach_key, stages, stages_by_approach)
    ]

    rows: list[dict[str, Any]] = []
    bert_candidates: list[str] = []
    bert_references: list[str] = []
    bert_row_indexes: list[int] = []

    total = len(approach_stage_pairs) * len(selected_questions)
    done = 0
    for approach_key, stage in approach_stage_pairs:
        for q_idx, item in enumerate(selected_questions, start=1):
            done += 1
            print(
                f"[{done}/{total}] {RAG_APPROACHES[approach_key]['label']} | "
                f"{retrieval_stage_label(approach_key, stage)} | Q{q_idx}"
            )
            row, bert_candidate, bert_reference = _build_eval_row(
                item=item,
                q_idx=q_idx,
                approach_key=approach_key,
                stage=stage,
                compute_bert_later=compute_bert,
            )
            if compute_bert and bert_candidate is not None and bert_reference is not None:
                bert_row_indexes.append(len(rows))
                bert_candidates.append(bert_candidate)
                bert_references.append(bert_reference)
            rows.append(row)

    if compute_bert and bert_candidates:
        bert_rows = compute_bertscore_batch(bert_candidates, bert_references)
        for idx, scores in zip(bert_row_indexes, bert_rows):
            rows[idx].update(scores)

    for row in rows:
        row.setdefault("bertscore_precision", math.nan)
        row.setdefault("bertscore_recall", math.nan)
        row.setdefault("bertscore_f1", math.nan)
        row.setdefault("bertscore_backend", "disabled" if not compute_bert else "failed")

    return pd.DataFrame(rows) if pd is not None else rows


def summarize_evaluation_results(detail_df):
    if pd is None:
        return detail_df
    if detail_df is None or len(detail_df) == 0:
        return pd.DataFrame()
    metric_cols = [
        "bertscore_f1", "precision", "recall", "answer_f1",
        "context_recall", "timestamp_coverage", "must_recall", "should_recall",
        "answer_claim_recall", "n_retrieved", "retrieved_seconds",
    ]
    return (
        detail_df
        .groupby(["approach", "retrieval_view"], dropna=False)[metric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values(["approach", "retrieval_view"])
    )


def evaluate_rag_approaches(
    test_set: list[dict] = TEST_SET,
    mode_by_approach: dict[str, str] | None = None,
    approaches: Iterable[str] | None = None,
    limit: int | None = None,
    compute_bert: bool = True,
) -> tuple[Any, Any]:
    approaches = list(approaches or RAG_APPROACHES.keys())
    mode_by_approach = mode_by_approach or {
        key: ("rerank" if "rerank" in APPROACH_RETRIEVAL_STAGES.get(key, []) else APPROACH_RETRIEVAL_STAGES.get(key, ["dense"])[0])
        for key in approaches
    }
    selected_questions = test_set[:limit] if limit else test_set

    rows: list[dict[str, Any]] = []
    bert_candidates: list[str] = []
    bert_references: list[str] = []
    bert_row_indexes: list[int] = []
    for approach_key in approaches:
        stage = mode_by_approach.get(approach_key, "rerank")
        if stage not in APPROACH_RETRIEVAL_STAGES.get(approach_key, []):
            continue
        for q_idx, item in enumerate(selected_questions, start=1):
            row, bert_candidate, bert_reference = _build_eval_row(
                item=item,
                q_idx=q_idx,
                approach_key=approach_key,
                stage=stage,
                compute_bert_later=compute_bert,
            )
            if compute_bert and bert_candidate is not None and bert_reference is not None:
                bert_row_indexes.append(len(rows))
                bert_candidates.append(bert_candidate)
                bert_references.append(bert_reference)
            rows.append(row)

    if compute_bert and bert_candidates:
        bert_rows = compute_bertscore_batch(bert_candidates, bert_references)
        for idx, scores in zip(bert_row_indexes, bert_rows):
            rows[idx].update(scores)
    for row in rows:
        row.setdefault("bertscore_precision", math.nan)
        row.setdefault("bertscore_recall", math.nan)
        row.setdefault("bertscore_f1", math.nan)
        row.setdefault("bertscore_backend", "disabled" if not compute_bert else "failed")

    if pd is None:
        return rows, rows
    detail_df = pd.DataFrame(rows)
    return summarize_evaluation_results(detail_df), detail_df


def display_evaluation_tables(summary_df, detail_df, show_answers: bool = False, show_bert: bool = True):
    if pd is None:
        display(summary_df)
        return
    metric_format = {
        "bertscore_precision": "{:.3f}",
        "bertscore_recall": "{:.3f}",
        "bertscore_f1": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "answer_f1": "{:.3f}",
        "context_recall": "{:.3f}",
        "timestamp_coverage": "{:.3f}",
        "must_recall": "{:.3f}",
        "should_recall": "{:.3f}",
        "answer_claim_recall": "{:.3f}",
        "n_retrieved": "{:.1f}",
        "retrieved_seconds": "{:.1f}",
    }
    summary_cols = ["approach", "retrieval_view"]
    detail_cols = ["question_id", "approach", "retrieval_view", "requires_timestamp"]
    if show_bert:
        summary_cols += ["bertscore_f1"]
        detail_cols += ["bertscore_f1", "bertscore_backend"]
    summary_cols += [
        "precision", "recall", "context_recall", "timestamp_coverage",
        "must_recall", "should_recall", "n_retrieved", "retrieved_seconds",
    ]
    detail_cols += [
        "precision", "recall", "context_recall", "timestamp_coverage",
        "must_covered", "should_covered", "context_claims_covered",
        "timestamp_targets_covered", "question",
    ]
    if show_answers:
        detail_cols += ["answer", "timestamp_detail"]
    summary_cols = [c for c in summary_cols if c in summary_df.columns]
    detail_cols = [c for c in detail_cols if c in detail_df.columns]
    display(summary_df[summary_cols].style.format(metric_format, na_rep="n/a"))
    display(detail_df[detail_cols].style.format(metric_format, na_rep="n/a"))

print("Metric functions ready: BERTScore via bert_score, Precision, Recall, Context Recall, Timestamp Coverage, Must/Should Recall.")


Metric functions ready: BERTScore via bert_score, Precision, Recall, Context Recall, Timestamp Coverage, Must/Should Recall.


### 6.4 Generate Evaluation Answers

Run this cell when you want to refresh the evaluation cache. It performs retrieval and answer generation once, then computes metrics. The interactive table in the next cell only filters this cached dataframe, so changing dropdowns does not call the generator again.


In [20]:
# Configure this cache build before running the cell.
# Increase EVAL_QUESTION_LIMIT to len(TEST_SET) for the full benchmark.
EVAL_APPROACHES_TO_RUN = ["whisper", "clap"]
EVAL_STAGES_BY_APPROACH = {
    "whisper": ["dense", "bm25", "hybrid", "rerank"],
    "clap": ["dense", "rerank"],
}
EVAL_QUESTION_LIMIT = min(4, len(TEST_SET))
EVAL_COMPUTE_BERTSCORE = True

EVAL_DETAIL_DF = generate_evaluation_results(
    approaches=EVAL_APPROACHES_TO_RUN,
    stages_by_approach=EVAL_STAGES_BY_APPROACH,
    limit=EVAL_QUESTION_LIMIT,
    compute_bert=EVAL_COMPUTE_BERTSCORE,
)
EVAL_SUMMARY_DF = summarize_evaluation_results(EVAL_DETAIL_DF)

print(
    f"Cached {len(EVAL_DETAIL_DF)} evaluation rows "
    f"({sum(len(EVAL_STAGES_BY_APPROACH.get(k, [])) for k in EVAL_APPROACHES_TO_RUN)} valid approach/retrieval views x {EVAL_QUESTION_LIMIT} questions)."
)
display_evaluation_tables(EVAL_SUMMARY_DF, EVAL_DETAIL_DF, show_answers=False, show_bert=True)


[1/24] Whisper transcript RAG | Whisper dense retrieval only | Q1
[2/24] Whisper transcript RAG | Whisper dense retrieval only | Q2
[3/24] Whisper transcript RAG | Whisper dense retrieval only | Q3
[4/24] Whisper transcript RAG | Whisper dense retrieval only | Q4
[5/24] Whisper transcript RAG | Whisper BM25 only | Q1
[6/24] Whisper transcript RAG | Whisper BM25 only | Q2
[7/24] Whisper transcript RAG | Whisper BM25 only | Q3
[8/24] Whisper transcript RAG | Whisper BM25 only | Q4
[9/24] Whisper transcript RAG | Whisper dense + BM25 | Q1
[10/24] Whisper transcript RAG | Whisper dense + BM25 | Q2
[11/24] Whisper transcript RAG | Whisper dense + BM25 | Q3
[12/24] Whisper transcript RAG | Whisper dense + BM25 | Q4
[13/24] Whisper transcript RAG | Whisper dense + BM25 + cross-encoder rerank | Q1
[14/24] Whisper transcript RAG | Whisper dense + BM25 + cross-encoder rerank | Q2
[15/24] Whisper transcript RAG | Whisper dense + BM25 + cross-encoder rerank | Q3
[16/24] Whisper transcript RAG | Wh

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cached 24 evaluation rows (6 valid approach/retrieval views x 4 questions).


,approach,retrieval_view,bertscore_f1,precision,recall,context_recall,timestamp_coverage,must_recall,should_recall,n_retrieved,retrieved_seconds
0,CLAP audio-text RAG,CLAP audio-text dense retrieval only,0.635,0.000,0.000,0.083,0.000,0.000,0.000,4.0,40.0
1,CLAP audio-text RAG,CLAP dense + transcript cross-encoder rerank,0.635,0.000,0.000,0.125,0.333,0.000,0.000,4.0,40.0
2,Whisper transcript RAG,Whisper BM25 only,0.635,0.000,0.000,0.125,0.333,0.000,0.000,4.0,110.9
3,Whisper transcript RAG,Whisper dense + BM25,0.635,0.000,0.000,0.042,0.333,0.000,0.000,4.0,111.1
4,Whisper transcript RAG,Whisper dense + BM25 + cross-encoder rerank,0.635,0.000,0.000,0.083,0.333,0.000,0.000,4.0,111.8
5,Whisper transcript RAG,Whisper dense retrieval only,0.635,0.000,0.000,0.042,0.333,0.000,0.000,4.0,111.1


,question_id,approach,retrieval_view,requires_timestamp,bertscore_f1,bertscore_backend,precision,recall,context_recall,timestamp_coverage,must_covered,should_covered,context_claims_covered,timestamp_targets_covered,question
0,1,Whisper transcript RAG,Whisper dense retrieval only,True,0.638,bert_score,0.000,0.000,0.167,1.000,0/3,0/3,1/6,1/1,Describe the overall architecture of the Transformer model
1,2,Whisper transcript RAG,Whisper dense retrieval only,True,0.638,bert_score,0.000,0.000,0.000,0.000,0/1,0/2,0/3,0/1,What BLEU scores did the Transformer achieve on WMT 2014 English-to-German translation?
2,3,Whisper transcript RAG,Whisper dense retrieval only,True,0.622,bert_score,0.000,0.000,0.000,n/a,0/2,0/2,0/4,0/0,What is the formula for scaled dot-product attention?
3,4,Whisper transcript RAG,Whisper dense retrieval only,True,0.643,bert_score,0.000,0.000,0.000,0.000,0/3,0/2,0/5,0/1,What is positional encoding and why is it necessary in the Transformer?
4,1,Whisper transcript RAG,Whisper BM25 only,True,0.638,bert_score,0.000,0.000,0.500,1.000,0/3,0/3,3/6,1/1,Describe the overall architecture of the Transformer model
5,2,Whisper transcript RAG,Whisper BM25 only,True,0.638,bert_score,0.000,0.000,0.000,0.000,0/1,0/2,0/3,0/1,What BLEU scores did the Transformer achieve on WMT 2014 English-to-German translation?
6,3,Whisper transcript RAG,Whisper BM25 only,True,0.622,bert_score,0.000,0.000,0.000,n/a,0/2,0/2,0/4,0/0,What is the formula for scaled dot-product attention?
7,4,Whisper transcript RAG,Whisper BM25 only,True,0.643,bert_score,0.000,0.000,0.000,0.000,0/3,0/2,0/5,0/1,What is positional encoding and why is it necessary in the Transformer?
8,1,Whisper transcript RAG,Whisper dense + BM25,True,0.638,bert_score,0.000,0.000,0.167,1.000,0/3,0/3,1/6,1/1,Describe the overall architecture of the Transformer model
9,2,Whisper transcript RAG,Whisper dense + BM25,True,0.638,bert_score,0.000,0.000,0.000,0.000,0/1,0/2,0/3,0/1,What BLEU scores did the Transformer achieve on WMT 2014 English-to-German translation?


### 6.5 Interactive Check Table

This widget reads `EVAL_DETAIL_DF` from the previous cell. It does not run retrieval, reranking, generation, or BERTScore again.


In [21]:
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as e:
    widgets = None
    WIDGETS_AVAILABLE = False
    print(f"ipywidgets unavailable: {e}")


def _cached_stage_options(detail_df, approach_key: str):
    stages = list(detail_df.loc[detail_df["approach_key"] == approach_key, "retrieval_stage"].dropna().unique())
    ordered = [stage for stage in RETRIEVAL_STAGE_OPTIONS if stage in stages]
    ordered += [stage for stage in stages if stage not in ordered]
    return [(retrieval_stage_label(approach_key, stage), stage) for stage in ordered]


def _filter_cached_evaluation(detail_df, selected: list[str], mode_by_approach: dict[str, str], question_limit: int):
    if pd is None:
        return detail_df, detail_df
    if detail_df is None or len(detail_df) == 0 or not selected:
        return pd.DataFrame(), pd.DataFrame()
    mask = pd.Series(False, index=detail_df.index)
    for key in selected:
        mask |= (
            (detail_df["approach_key"] == key)
            & (detail_df["retrieval_stage"] == mode_by_approach.get(key))
        )
    filtered = detail_df.loc[mask & (detail_df["question_id"] <= question_limit)].copy()
    return summarize_evaluation_results(filtered), filtered


def make_rag_evaluation_dashboard(detail_df=None):
    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available. Use EVAL_DETAIL_DF directly or install ipywidgets.")
        return None
    if pd is None:
        print("pandas is not available. The cached dashboard requires pandas dataframes.")
        return None
    if detail_df is None:
        detail_df = globals().get("EVAL_DETAIL_DF")
    if detail_df is None or len(detail_df) == 0:
        print("Run the previous 'Generate Evaluation Answers' cell first to create EVAL_DETAIL_DF.")
        return None

    cached_approaches = [key for key in RAG_APPROACHES if key in set(detail_df["approach_key"])]
    approach_checks = {
        key: widgets.Checkbox(value=True, description=RAG_APPROACHES[key]["label"], indent=False)
        for key in cached_approaches
    }
    mode_dropdowns = {}
    for key in cached_approaches:
        options = _cached_stage_options(detail_df, key)
        default = "rerank" if any(value == "rerank" for _, value in options) else options[0][1]
        mode_dropdowns[key] = widgets.Dropdown(
            options=options,
            value=default,
            layout=widgets.Layout(width="330px"),
        )

    max_cached_questions = int(detail_df["question_id"].max())
    question_limit = widgets.IntSlider(
        value=max_cached_questions,
        min=1,
        max=max_cached_questions,
        step=1,
        description="Questions",
        continuous_update=False,
        layout=widgets.Layout(width="360px"),
    )
    show_bert = widgets.Checkbox(value=True, description="Show BERTScore", indent=False)
    show_answers = widgets.Checkbox(value=False, description="Show answers", indent=False)
    update_button = widgets.Button(description="Update table", button_style="primary", icon="filter")
    output = widgets.Output()

    rows = [widgets.HTML("<b>RAG approach</b>"), widgets.HTML("<b>Retrieval view</b>")]
    for key in cached_approaches:
        rows.append(approach_checks[key])
        rows.append(mode_dropdowns[key])
    selector_grid = widgets.GridBox(
        rows,
        layout=widgets.Layout(
            grid_template_columns="minmax(330px, 1fr) 350px",
            grid_gap="8px 12px",
            align_items="center",
        ),
    )

    def _render(_=None):
        selected = [key for key, check in approach_checks.items() if check.value]
        mode_by_approach = {key: mode_dropdowns[key].value for key in selected}
        with output:
            clear_output(wait=True)
            if not selected:
                print("Select at least one cached RAG approach.")
                return
            summary_df, filtered_df = _filter_cached_evaluation(
                detail_df=detail_df,
                selected=selected,
                mode_by_approach=mode_by_approach,
                question_limit=question_limit.value,
            )
            if filtered_df.empty:
                print("No cached rows for this selection. Rerun the previous cell with those retrieval views enabled.")
                return
            display_evaluation_tables(
                summary_df,
                filtered_df,
                show_answers=show_answers.value,
                show_bert=show_bert.value,
            )

    update_button.on_click(_render)
    for widget in [question_limit, show_bert, show_answers, *approach_checks.values(), *mode_dropdowns.values()]:
        widget.observe(_render, names="value")

    ui = widgets.VBox([
        widgets.HTML("<h3>Interactive Audio RAG Evaluation Table</h3>"),
        widgets.HTML("<i>Cached view: changing filters does not regenerate answers.</i>"),
        selector_grid,
        widgets.HBox([question_limit, show_bert, show_answers, update_button]),
        output,
    ])
    display(ui)
    _render()
    return ui


evaluation_dashboard = make_rag_evaluation_dashboard()


### 6.6 Answer Generation Showcase

Use this final cell to generate side-by-side answer examples after the audio indexes and generator are loaded. The examples reuse the same approach/retrieval controls as the evaluation table.


In [22]:
SHOWCASE_QUESTIONS = [
    "What is the formula for scaled dot-product attention?",
    "Describe the overall architecture of the Transformer model",
    "How is BERT positioned in regard to transformers?",
]


def _doc_preview(doc: Any, max_chars: int = 220) -> str:
    ts = getattr(doc, "timestamp_label", "")
    text = re.sub(r"\s+", " ", getattr(doc, "content", "")).strip()
    return f"{ts} " + text[:max_chars]


def showcase_answer_generation(
    questions: list[str] = SHOWCASE_QUESTIONS,
    approaches: Iterable[str] = ("whisper", "clap"),
    mode_by_approach: dict[str, str] | None = None,
    max_sources: int = 3,
):
    mode_by_approach = mode_by_approach or {key: "rerank" for key in approaches}
    rows = []
    for question in questions:
        for approach_key in approaches:
            spec = RAG_APPROACHES[approach_key]
            stage = mode_by_approach.get(approach_key, "rerank")
            docs = spec["retriever"](question, stage=stage)
            answer = generate_answer(docs, question)
            rows.append({
                "question": question,
                "approach": spec["label"],
                "retrieval_view": RETRIEVAL_STAGE_OPTIONS.get(stage, stage),
                "answer": answer,
                "sources": "\n".join(_doc_preview(d) for d in docs[:max_sources]),
            })

    if pd is not None:
        df = pd.DataFrame(rows)
        display(df.style.set_properties(subset=["answer", "sources"], **{"white-space": "pre-wrap"}))
    elif HTML is not None:
        html_rows = []
        for row in rows:
            html_rows.append(
                "<tr>"
                f"<td>{escape(row['question'])}</td>"
                f"<td>{escape(row['approach'])}</td>"
                f"<td>{escape(row['retrieval_view'])}</td>"
                f"<td><pre>{escape(row['answer'])}</pre></td>"
                f"<td><pre>{escape(row['sources'])}</pre></td>"
                "</tr>"
            )
        display(HTML("<table>" + "".join(html_rows) + "</table>"))
    else:
        for row in rows:
            print("=" * 100)
            print(row["question"])
            print(row["approach"], "|", row["retrieval_view"])
            print(row["answer"])
            print(row["sources"])


showcase_answer_generation(
    questions=SHOWCASE_QUESTIONS,
    mode_by_approach={
        "whisper": "rerank",
        "clap": "rerank",
    },
)


,question,approach,retrieval_view,answer,sources
0,What is the formula for scaled dot-product attention?,Whisper transcript RAG,Cross-encoder rerank,[Generator unavailable],"[06:19 – 06:48] BERT's model architecture is a multi-layer bi-directional transformer encoder. In this work, we primarily report results on two model sizes, BERT-BASE with 110 million parameters and BERT-LARGE with 340 million parameter [06:48 – 07:18] while the GPT transformer uses constrained self-attention where every token can only attend to context to its left. Input and Output Representations To make BERT handle a variety of downstream tasks, our input representa [09:10 – 09:37] For the pre-training corpus, we use the books corpus and English Wikipedia, extracting only the text passages to ensure the model learns from long contiguous sequences. Fine-tuning BERT. Fine-tuning is straightforward si"
1,What is the formula for scaled dot-product attention?,CLAP audio-text RAG,Cross-encoder rerank,[Generator unavailable],"[06:32 – 06:42] BERT's model architecture is a multi-layer bi-directional transformer encoder. In this work, we primarily report results on two model sizes, BERT-BASE with 110 million parameters and BERT-LARGE with 340 million parameter [02:48 – 02:58] The authors use a left-to-right architecture, where every token can only attend to previous tokens in the self-attention layers of the transformer. Such restrictions are suboptimal for sentence-level tasks. It can be ver [09:52 – 10:02] We plug in the task's specific inputs and outputs into BERT and fine-tune all the parameters end-to-end. Compared to pre-training, fine-tuning is relatively inexpensive. Experimental results and conclusion. Extensive exp"
2,Describe the overall architecture of the Transformer model,Whisper transcript RAG,Cross-encoder rerank,[Generator unavailable],"[06:19 – 06:48] BERT's model architecture is a multi-layer bi-directional transformer encoder. In this work, we primarily report results on two model sizes, BERT-BASE with 110 million parameters and BERT-LARGE with 340 million parameter [01:47 – 02:14] uses task-specific architectures that include the pre-trained representations as additional features. The fine-tuning approach, such as the Generative Pre-Trained Transformer or OpenAI GPT, introduces minimal task-specif [02:37 – 03:05] The authors use a left-to-right architecture, where every token can only attend to previous tokens in the self-attention layers of the transformer. Such restrictions are suboptimal for sentence-level tasks. It can be ver"
3,Describe the overall architecture of the Transformer model,CLAP audio-text RAG,Cross-encoder rerank,[Generator unavailable],"[06:32 – 06:42] BERT's model architecture is a multi-layer bi-directional transformer encoder. In this work, we primarily report results on two model sizes, BERT-BASE with 110 million parameters and BERT-LARGE with 340 million parameter [02:48 – 02:58] The authors use a left-to-right architecture, where every token can only attend to previous tokens in the self-attention layers of the transformer. Such restrictions are suboptimal for sentence-level tasks. It can be ver [06:08 – 06:18] During pre-training, the model is trained on unlabeled data over different pre-training tasks. For fine-tuning, the BERT model is first initialized with the pre-trained parameters, and all of the parameters are fine-tune"
4,How is BERT positioned in regard to transformers?,Whisper transcript RAG,Cross-encoder rerank,[Generator unavailable],"[00:00 – 00:25] BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language representation mode [06:19 – 06:48] BERT's model architecture is a multi-layer bi-directional transformer encoder. In this work, we primarily report results on two model sizes, BERT-BASE with 110 million parameters and BERT-LARGE with 340 million parameter [09: